In [7]:
import sys; sys.path.append("..")
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_parquet("../data/processed/phrasebank.parquet")
print(df["label"].value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"])
print(len(X_train), "train |", len(X_test), "test")

label
neutral     2141
positive     887
negative     420
Name: count, dtype: int64
2758 train | 690 test


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
tfidf_lr.fit(X_train, y_train)
pred_lr = tfidf_lr.predict(X_test)
print(classification_report(y_test, pred_lr, digits=3))

              precision    recall  f1-score   support

    negative      0.701     0.643     0.671        84
     neutral      0.863     0.909     0.885       428
    positive      0.735     0.669     0.700       178

    accuracy                          0.814       690
   macro avg      0.766     0.740     0.752       690
weighted avg      0.810     0.814     0.811       690



In [9]:
from transformers import pipeline

finbert = pipeline("text-classification", model="ProsusAI/finbert")
print(finbert("Quarterly revenue rose 20% beating all estimates."))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'positive', 'score': 0.9591554999351501}]


In [10]:
raw = finbert(list(X_test), truncation=True, batch_size=16)
pred_fb = [r["label"].lower() for r in raw]     # FinBERT returns 'Positive' etc → lowercase to match

print(set(pred_fb), "vs", set(y_test))          # sanity: label vocabularies must match
print(classification_report(y_test, pred_fb, digits=3))

{'positive', 'negative', 'neutral'} vs {'positive', 'negative', 'neutral'}
              precision    recall  f1-score   support

    negative      0.891     0.976     0.932        84
     neutral      0.993     0.930     0.960       428
    positive      0.878     0.972     0.923       178

    accuracy                          0.946       690
   macro avg      0.921     0.959     0.938       690
weighted avg      0.951     0.946     0.947       690



In [11]:
comp = pd.DataFrame({"text": X_test.values, "true": y_test.values,
                     "lr": pred_lr, "finbert": pred_fb})
disagree = comp[comp["lr"] != comp["finbert"]]
print(f"{len(disagree)} disagreements of {len(comp)}")
disagree.sample(10)

143 disagreements of 690


,text,true,lr,finbert
524,Aldata to Share Space Optimization Vision at A...,neutral,neutral,positive
391,`` We have analyzed Kaupthing Bank Sweden and ...,positive,neutral,positive
590,The commission said the hydrogen peroxide and ...,neutral,positive,neutral
264,According to ACNielsen 's ScanTrack study for ...,positive,neutral,positive
314,Finnish flexible packaging manufacturer Suomin...,positive,negative,positive
36,The hosting mobile terminal guides information...,neutral,positive,neutral
333,`` We continued actively to focus R&D and to p...,positive,neutral,positive
345,According to Deputy MD Pekka Silvennoinen the ...,positive,neutral,positive
303,Cameco typically prices sales contracts using ...,neutral,negative,neutral
32,"According to the company , in addition to norm...",negative,neutral,negative


## Day 12 verdict — sentiment
- TF-IDF + LogReg: macro-F1 **0.752**. FinBERT: macro-F1 **0.938** (+19 pts).
- FinBERT's negative-class recall 0.976 — it catches the rare, valuable bad-news class.
- Disagreements: FinBERT wins on finance idioms + negation (context it understands, bag-of-words can't).
- Contrast with the stock saga: ~0.51 (unlearnable direction) vs 0.94 (learnable sentiment)
  from ONE pipeline — high scores where signal is real, honest nulls where it isn't.